In [2]:
import os
import json
import requests
from tqdm import tqdm
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import islice

In [3]:
# ==============================
# CONFIGURATION
# ==============================
SAVE_DIR = "laion_images_kp"
METADATA_FILE = "laion_metadata_kp.jsonl"
N_SAMPLES = 1_000_000       # total samples you want
MAX_WORKERS = 32            # number of parallel download threads
TIMEOUT = 5                 # timeout per image in seconds

In [4]:
# ==============================
# SETUP
# ==============================
os.makedirs(SAVE_DIR, exist_ok=True)
dataset = load_dataset("laion/relaion2B-en-research-safe", streaming=True, split="train")
subset = dataset.shuffle(seed=42).take(N_SAMPLES)
subset = subset.remove_columns([
    'similarity', 'hash', 'pwatermark', 'punsafe', 'key',
    'status', 'error_message', 'width', 'height', 'original_width',
    'original_height', 'exif', 'md5'
])

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

In [5]:
subset

IterableDataset({
    features: ['url', 'caption'],
    num_shards: 128
})

In [6]:
# ==============================
# DOWNLOAD FUNCTION
# ==============================
def download_image(idx, url):
    """Download image from URL and save locally."""
    try:
        response = requests.get(url, timeout=TIMEOUT)
        if response.status_code == 200 and response.content.startswith(b'\xff\xd8'):  # JPEG check
            path = os.path.join(SAVE_DIR, f"{idx}.jpg")
            with open(path, "wb") as f:
                f.write(response.content)
            return True
    except Exception:
        pass
    return False

In [7]:
# ==============================
# MAIN PIPELINE
# ==============================
success_count = 0
with open(METADATA_FILE, "w", encoding="utf-8") as meta_f, \
     ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {}
    for idx, sample in enumerate(tqdm(islice(subset, N_SAMPLES), total=N_SAMPLES, desc="Submitting jobs")):
        # print(sample)
        url = sample.get('url')
        print(url)
        text = sample.get('caption', "")
        print(text)
        if not url:
            print("null")
            continue

        # # Submit async download
        # futures[executor.submit(download_image, idx, url)] = (idx, url, text)

        # # Write metadata immediately (whether downloaded or not)
        # json.dump({"id": idx, "url": url, "text": text}, meta_f)
        # meta_f.write("\n")

    # Collect results
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading images"):
        idx, url, text = futures[future]
        if future.result():
            success_count += 1

print(f"✅ Completed. Downloaded {success_count} images successfully to '{SAVE_DIR}'")
print(f"Metadata saved in '{METADATA_FILE}'")


Submitting jobs:   0%|          | 439/1000000 [00:06<2:34:21, 107.93it/s]

https://en.wahooart.com/Art.nsf/O/8YEAYK/$File/Mary-Stevenson-Cassatt-Young-Woman-Sewing-in-the-Garden-S.JPG
Mary Stevenson Cassatt - Young Woman Sewing in the Garden
https://images.fun.com.au/products/26900/1-2/deluxe-inflatable-adult-godzilla-costume.jpg
Deluxe Inflatable Adult Godzilla Costume
https://slimages.macysassets.com/is/image/MCY/products/1/optimized/9845521_fpx.tif?$thumb$
JM Collection Handkerchief-Hem Maxi Skirt, Created for Macy's
https://thumbs1.ebaystatic.com/pict/3911144082964040_1.jpg
For LG Phone New Synthetic Leather Vintage ID Card Holder Media Stand Case Cover
https://thumbs.dreamstime.com/t/unusual-cave-houses-los-banos-near-guadix-spain-numerous-built-hillside-small-hill-andalucia-61569407.jpg
Unusual cave-houses in Los Banos near Guadix, Spain Royalty Free Stock Photography
https://cdn.shopify.com/s/files/1/1342/1165/products/product-image-110313116_compact.jpg?v=1571439717
Fabulous Women watches Retro Design Leather Band simple design Analog Alloy Quartz Wri

Submitting jobs:   0%|          | 804/1000000 [00:07<1:03:00, 264.28it/s]

https://images3.garmentory.com/images/1619339/medium/V-Neck-Knit-Pullover-20180914222300.jpg?1536963783
Closed V Neck Knit Pullover - Pearl
https://www.teachingtraveling.com/wp-content/uploads/2012/03/lgdiving.jpg
Lisa was proposed to underwater in Thailand!
https://rlv.zcache.ca/mandatory_retirement_age_51_for_obama_bumper_sticker-r65c307d4dbb34b46b39f1203d728f6e3_v9wht_8byvr_324.jpg
MANDATORY RETIREMENT AGE: 51, ...for Obama! Bumper Sticker
https://merchbar.imgix.net/product/69/5855/1994191667259/TmM6dWrwtheweeknd-kissland.png?w=390&h=390&quality=60&auto=compress%252Cformat
The Weeknd, Kiss Land (Limited Edition 2LP) (Vinyl)
http://www.whykol.com/wp-content/uploads/2016/07/PSC-Trolls171-300x300.jpg
Malayalam PSC Trolls | Gk as Jokes | Kerala PSC Questions
http://slideplayer.com/slide/1510885/5/images/22/Semiconductor+Devices.jpg
Semiconductor Devices
https://cdn.shoplightspeed.com/shops/632748/files/16648168/800x1024x2/the-fairhope-store-unisex-cap.jpg
The Fairhope Store Unisex Cap
h

Submitting jobs:   0%|          | 1273/1000000 [00:07<27:49, 598.14it/s] 

...

Submitting jobs:   2%|▏         | 19452/1000000 [00:14<06:15, 2609.85it/s]

https://www.barbadosbarbados.com/wp-content/uploads/2019/08/leamington-house-luxury-villa-rental-barbados-150x101.jpg
elsewhere-luxury-villa-rental-barbados
https://i.pinimg.com/736x/ee/f9/3d/eef93da2543dba080d5838b2c1b59eeb.jpg
Industrial Garment Rack Hanging Possum Belly by stellableudesigns
https://us.123rf.com/450wm/nameinframe1/nameinframe11512/nameinframe1151200608/49448947-exclamation-point-in-the-alphabet-set-country-lane-two-is-pink-with-black-outline--letter-sits-on-ar.jpg?ver=6
exclamation point: Exclamation Point, in the alphabet set Country Lane Two is pink with black outline.  Letter sits on arrangement of country flowers in pink and blue.
https://ca.egoshoes.com/media/catalog/product/f/1/f119-1.jpg?width=500
Maxine Long Boot In Black Faux Suede
https://i2.wp.com/www.wearetravellers.nl/wp-content/uploads/Top-of-the-rock-new-york-central-park-.jpg?w=246&
Top-of-the-rock-new-york-central-park
https://cdnimages.opentip.com/thumbs/BCW/BCW-970957_130_130.jpg
Mars Fishcare Nort

Submitting jobs:   2%|▏         | 19602/1000000 [00:14<11:59, 1363.49it/s]


https://www.cens.com/cens/supplier/3988/product/163907/MED.jpg?xxx=1521122886679
Impact Sockets,Pneumatic Tools, electric Tools,Sockets, Nuts
http://thumbs.slideserve.com/1_3343247.jpg
Physics of Information Technology
http://t0.gstatic.com/images?q=tbn:ANd9GcRfvso_tjp47uBal9Gx-Z4Yb1tL7nktoiYfqGJDTt3TJnA3YmdQOw
layout of a excel worksheet can u0027t print the entire worksheet
https://www.specsserver.com/CACHE/FRYKFWSJVNGI.JPG?width=220&amp;height=220&ccid=x15130a76
Café ENERGY STAR ® 27.8 Cu. Ft. French-Door Refrigerator with Keurig ® K-Cup ® Brewing System
https://render.fineartamerica.com/images/rendered/wall-view/medium/room002/print-poster/images/artworkimages/medium/1/mardi-gras-bruce-combs.jpg?printWidth=5.125&printHeight=8.000
Wall View 002
http://archive.li/e5dXA/46be4a91c99326e12d851339ffdef2e21163a567.jpg
Dawkins at the University of Texas at Austin.
https://prodold.vinculumgroup.com/wp-content/uploads/2016/09/proyog_237x195.jpg
Proyog partners with Vinculum to promote Authen

KeyboardInterrupt: 